# Data Loader

> Load and parse `data/data.json` from greeny/SatisfactoryTools into clean Python dicts.
>
> Exports: `load_data`, `get_item_recipe`

In [1]:
#| default_exp data

In [2]:
#| export
import json
from pathlib import Path

## Load raw data

In [3]:
#| export
def load_data(data_path: str | Path = None) -> dict:
    """Load data.json and return the raw dict with keys: items, recipes, buildings, etc.
    
    If data_path is not provided, resolves relative to this file's location:
    walks up to find the repo root (where data/data.json lives).
    """
    if data_path is None:
        # Walk up from this file to find data/data.json
        here = Path(__file__).resolve().parent
        # Try up to 3 levels up
        for _ in range(4):
            candidate = here / 'data' / 'data.json'
            if candidate.exists():
                data_path = candidate
                break
            here = here.parent
        else:
            raise FileNotFoundError('data/data.json not found. Pass data_path explicitly.')
    with open(data_path, encoding='utf-8') as f:
        return json.load(f)

## Wiki image URL helper

In [4]:
#| export
def wiki_image_url(item_name: str, size: int = 128) -> str:
    """Return the wiki.gg thumbnail URL for an item by display name.
    
    Pattern: https://satisfactory.wiki.gg/images/thumb/{Name}.png/{size}px-{Name}.png
    Spaces are replaced with underscores.
    """
    slug = item_name.replace(' ', '_')
    return f'https://satisfactory.wiki.gg/images/thumb/{slug}.png/{size}px-{slug}.png'

## Local image cache

In [ ]:
#| export
def image_slug(name: str) -> str:
    """Convert item display name to a filesystem-safe slug.

    Must match scripts/download_images.py exactly.
    """
    return name.replace(' ', '_').replace("'", '_').replace(':', '_')

In [ ]:
#| export
# ⚡ Resolve once at import time — works for both local dev and stlite
_STATIC_DIR: Path | None = None

def _find_static_dir() -> Path | None:
    """Locate app/static/images/ by walking up from this file."""
    here = Path(__file__).resolve().parent
    for _ in range(4):
        candidate = here / 'app' / 'static' / 'images'
        if candidate.is_dir():
            return candidate
        here = here.parent
    return None

_STATIC_DIR = _find_static_dir()


def local_image_url(item_name: str, size: int = 64) -> str:
    """Return local static URL if the image exists, else fall back to wiki.gg.

    Local path served by Streamlit: /app/static/images/{slug}.png
    The `size` parameter is ignored for local images (all cached at 64px)
    but passed through to wiki_image_url() as a fallback.
    """
    slug = image_slug(item_name)
    if _STATIC_DIR is not None and (_STATIC_DIR / f'{slug}.png').exists():
        return f'/app/static/images/{slug}.png'
    return wiki_image_url(item_name, size)

## Item recipe lookup

In [5]:
#| export
def get_item_recipe(item_name: str, data: dict, alternate: bool = False) -> dict | None:
    """Look up an item by display name and return a clean recipe dict.

    Returns a dict with:
        name          -- item display name
        description   -- item description
        image_url     -- wiki.gg thumbnail URL
        ingredients   -- list of {name, amount, rate_per_min}
        products      -- list of {name, amount, rate_per_min}
        machine       -- building display name
        machine_power -- power draw in MW
        cycle_time    -- seconds per cycle
        recipe_name   -- recipe display name
        alternate     -- True if this is an alternate recipe

    If alternate=False (default), returns the standard (non-alternate) recipe.
    If alternate=True, returns the first alternate recipe found.
    Returns None if no matching item or recipe is found.
    """
    items = data.get('items', {})
    recipes = data.get('recipes', {})
    buildings = data.get('buildings', {})

    # Find item class key by display name
    item_key = None
    item_data = None
    for key, item in items.items():
        if item.get('name', '').lower() == item_name.lower():
            item_key = key
            item_data = item
            break
    if item_key is None:
        return None

    # Find matching recipe (standard or alternate)
    recipe_data = None
    for recipe in recipes.values():
        if not recipe.get('inMachine', False):
            continue
        products = recipe.get('products', [])
        if not any(p['item'] == item_key for p in products):
            continue
        is_alt = recipe.get('alternate', False)
        if is_alt == alternate:
            recipe_data = recipe
            break

    if recipe_data is None:
        return None

    cycle_time = recipe_data.get('time', 1)  # seconds
    cycles_per_min = 60.0 / cycle_time

    def resolve_ingredient(entry):
        """Resolve an ingredient/product entry to {name, amount, rate_per_min}."""
        ref_key = entry['item']
        amount = entry['amount']
        # Look up in items first, then resources
        ref_item = items.get(ref_key) or data.get('resources', {}).get(ref_key, {})
        return {
            'name': ref_item.get('name', ref_key),
            'amount': amount,
            'rate_per_min': round(amount * cycles_per_min, 4),
        }

    ingredients = [resolve_ingredient(e) for e in recipe_data.get('ingredients', [])]
    products = [resolve_ingredient(e) for e in recipe_data.get('products', [])]

    # Resolve machine
    produced_in = recipe_data.get('producedIn', [])
    machine_key = produced_in[0] if produced_in else None
    machine_data = buildings.get(machine_key, {}) if machine_key else {}
    machine_name = machine_data.get('name', machine_key or 'Unknown')
    machine_power = machine_data.get('metadata', {}).get('powerConsumption', 0)

    return {
        'name': item_data.get('name', item_name),
        'description': item_data.get('description', ''),
        'image_url': local_image_url(item_data.get('name', item_name)),
        'ingredients': ingredients,
        'products': products,
        'machine': machine_name,
        'machine_power': machine_power,
        'cycle_time': cycle_time,
        'recipe_name': recipe_data.get('name', ''),
        'alternate': recipe_data.get('alternate', False),
    }

## Test: all 6 target items

In [ ]:
# Not exported — test only
from pathlib import Path
from IPython.display import HTML, display

DATA_PATH = Path('../data/data.json')
data = load_data(DATA_PATH)

def recipe_card(result: dict) -> str:
    """Return an HTML crafting card styled like the Satisfactory wiki."""
    name = result['name']
    img = result['image_url']
    machine = result['machine']
    power = result['machine_power']
    cycle = result['cycle_time']
    recipe_name = result['recipe_name']

    def item_cell(entries):
        parts = []
        for e in entries:
            e_img = wiki_image_url(e['name'], 48)
            parts.append(
                f'<div style="display:inline-flex;align-items:center;gap:4px;margin:4px 6px;">'
                f'<span style="font-weight:600;color:#e8d44d;">{e["amount"]}&times;</span>'
                f'<img src="{e_img}" width="40" height="40" '
                f'style="image-rendering:pixelated;border:1px solid #555;border-radius:4px;background:#1a1a2e;">'
                f'<span style="font-size:0.82em;color:#ccc;">{e["name"]}<br>'
                f'<span style="color:#7ec8e3;">{e["rate_per_min"]}/min</span></span>'
                f'</div>'
            )
        return ''.join(parts)

    return f'''
    <div style="background:linear-gradient(135deg,#0f0f23,#1a1a2e);border:1px solid #e8d44d;
                border-radius:10px;margin:12px 0;padding:0;font-family:Segoe UI,sans-serif;
                color:#eee;overflow:hidden;max-width:820px;">
      <div style="background:linear-gradient(90deg,#e8d44d,#d4a017);padding:8px 16px;
                  display:flex;align-items:center;gap:12px;">
        <img src="{img}" width="48" height="48"
             style="image-rendering:pixelated;border:2px solid #fff;border-radius:6px;background:#1a1a2e;">
        <div>
          <div style="font-size:1.2em;font-weight:700;color:#0f0f23;">{name}</div>
          <div style="font-size:0.8em;color:#333;">Recipe: {recipe_name}</div>
        </div>
      </div>
      <table style="width:100%;border-collapse:collapse;margin:0;">
        <tr style="background:#12122a;">
          <th style="padding:8px 12px;text-align:center;border-bottom:1px solid #333;color:#e8d44d;
                     font-size:0.85em;width:40%;">Ingredients</th>
          <th style="padding:8px 12px;text-align:center;border-bottom:1px solid #333;color:#e8d44d;
                     font-size:0.85em;width:25%;">Produced in</th>
          <th style="padding:8px 12px;text-align:center;border-bottom:1px solid #333;color:#e8d44d;
                     font-size:0.85em;width:35%;">Products</th>
        </tr>
        <tr>
          <td style="padding:10px 12px;vertical-align:middle;border-right:1px solid #222;">
            {item_cell(result['ingredients'])}
          </td>
          <td style="padding:10px 12px;text-align:center;vertical-align:middle;border-right:1px solid #222;">
            <div style="font-weight:600;font-size:1.05em;color:#fff;">{machine}</div>
            <div style="font-size:0.82em;color:#7ec8e3;">{cycle}s cycle &bull; {power} MW</div>
          </td>
          <td style="padding:10px 12px;vertical-align:middle;">
            {item_cell(result['products'])}
          </td>
        </tr>
      </table>
    </div>'''

targets = [
    'Versatile Framework',
    'Modular Engine',
    'Adaptive Control Unit',
    'Heavy Modular Frame',
    'Automated Wiring',
    'Cable',
]

html_parts = []
for name in targets:
    result = get_item_recipe(name, data)
    if result is None:
        html_parts.append(f'<div style="color:rso for part of that i mean especially for the items maybe let's start with items but we need to in the maybe in the um notebook do something which allows us to list you know every single item there is within the item tab we'll list all of them and then that will also need to have a search on it um you know keep it fairly lightweight needs to have an icon a certain instant search needs to have a little icon the name um yeah and then when you click it um what i want to turn up is like what we can already see that little um that little modal thing that we've that we've already built in the uh i don't know what it's called let me have a look it is called get item recipe i think yeah get item recipe so we're talking about items here not resourcesed;padding:8px;">FAIL: {name} — not found</div>')
    else:
        html_parts.append(recipe_card(result))

display(HTML(''.join(html_parts)))

Ingredients,Produced in,Products
1.0×Modular Frame2.5/min12.0×Steel Beam30.0/min,Assembler 24s cycle • 15 MW,2.0×Versatile Framework5.0/min
Ingredients,Produced in,Products
2.0×Motor2.0/min15.0×Rubber15.0/min2.0×Smart Plating2.0/min,Manufacturer 60s cycle • 55 MW,1.0×Modular Engine1.0/min
Ingredients,Produced in,Products
15.0×Automated Wiring7.5/min10.0×Circuit Board5.0/min2.0×Heavy Modular Frame1.0/min2.0×Computer1.0/min,Manufacturer 120s cycle • 55 MW,2.0×Adaptive Control Unit1.0/min
Ingredients,Produced in,Products
5.0×Modular Frame10.0/min15.0×Steel Pipe30.0/min5.0×Encased Industrial Beam10.0/min100.0×Screw200.0/min,Manufacturer 30s cycle • 55 MW,1.0×Heavy Modular Frame2.0/min
Ingredients,Produced in,Products
1.0×Stator2.5/min20.0×Cable50.0/min,Assembler 24s cycle • 15 MW,1.0×Automated Wiring2.5/min
Ingredients,Produced in,Products


## List all craftable items

In [ ]:
#| export
def list_items(data: dict, size: int = 40) -> list[dict]:
    """Return all craftable items sorted A-Z, excluding raw resources.

    Each entry is a dict with:
        class_key  -- internal class key from data.json
        name       -- display name
        image_url  -- local static URL (falls back to wiki.gg)

    Raw resources (data['resources']) are excluded — these are ore, water,
    etc. that cannot be crafted and have no recipe.
    """
    resource_keys = set(data.get('resources', {}).keys())
    items = [
        {
            'class_key': key,
            'name': item.get('name', key),
            'image_url': local_image_url(item.get('name', key)),
        }
        for key, item in data.get('items', {}).items()
        if key not in resource_keys
    ]
    return sorted(items, key=lambda x: x['name'].lower())

## Test: list_items()

In [ ]:
# Not exported — test only
from pathlib import Path
DATA_PATH = Path('../data/data.json')
data_test = load_data(DATA_PATH)
items = list_items(data_test)
print(f'Total craftable items: {len(items)}')
print('First 5:', [i['name'] for i in items[:5]])
print('Last 5: ', [i['name'] for i in items[-5:]])
# Verify no resources included
resource_names = {v.get('name') for v in data_test.get('resources', {}).values()}
overlap = [i['name'] for i in items if i['name'] in resource_names]
print('Resource overlap (should be []):', overlap)

In [8]:
import nbdev; nbdev.nbdev_export()